# 02 - Dataset2 Linear SVM Classification Validation

Bu notebook, Dataset2 için kaydedilmiş LinearSVC modellerini valid split üzerinde değerlendirir.

Amaç, model performans tablolarını ve karşılaştırma görsellerini oluşturarak sonraki model seçimi adımına temel sağlamaktır.


## 1. Kütüphaneler

Bu bölümde model dosyalarını yüklemek, parquet feature tablolarını okumak, metrik hesaplamak ve görselleştirme yapmak için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = "pyarrow"
    print(f"[OK] pyarrow available: {pyarrow.__version__}")
except ImportError as exc:
    raise ImportError(
        "pyarrow is required for reading Stage 03 parquet feature files. "
        "Install it with: conda install -c conda-forge pyarrow"
    ) from exc


## 2. Ayarlar

Dataset bilgileri, validasyon split'i, sıralama metriği, liste boyutları ve overwrite davranışı tanımlanır.


In [ ]:
DATASET_NAME = "dataset2_honeybee"
DATASET_DISPLAY_NAME = "Dataset2 Honeybee"
DATASET_SHORT = "d2"

STAGE_NAME = "05_linear_svm_classification_validation"
NOTEBOOK_NAME = "02_dataset2_linear_svm_classification_validation"

RANDOM_STATE = 42

# Bu notebook yalnızca valid split üzerinde değerlendirme yapar.
EVALUATION_SPLIT = "valid"
PRIMARY_RANKING_METRIC = "valid_f1"

# İnceleme listesi boyutları
OVERALL_TOP_N = 20
RATIO_TOP_N = 10

# Model ve feature dosyaları bu notebookta yazılmaz.
OVERWRITE_CLASSIFICATION_RESULTS = False
OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

PATCH_SETS = ['centered48_balanced', 'centered48_neg4x', 'centered64_balanced', 'centered64_neg4x', 'centered80_balanced', 'centered80_neg4x']

DISPLAY_PREVIEW_ROWS = 10
REPORT_MAX_TABLE_ROWS = 50

NOTEBOOK_STARTED_AT = datetime.now()

print(f"[START] {NOTEBOOK_NAME} at {NOTEBOOK_STARTED_AT:%Y-%m-%d %H:%M:%S}")
print(f"[INFO] Dataset: {DATASET_NAME}")
print(f"[INFO] Evaluation split: {EVALUATION_SPLIT}")
print(f"[INFO] Primary ranking metric: {PRIMARY_RANKING_METRIC}")
print(f"[INFO] Expected patch sets: {len(PATCH_SETS)}")


## 3. Yardımcı Fonksiyonlar ve Dosya Yolları

Proje kökü, feature manifest dosyası, Linear SVM model klasörü ve bu notebooka ait tablo/görsel çıktı klasörleri hazırlanır.


In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()

    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "approaches").exists():
            return candidate

    raise FileNotFoundError(
        "Proje kökü bulunamadı. Notebook'u proje klasörü içinde çalıştırdığından emin ol."
    )


def resolve_project_path(path_value, fallback_base=None):
    if path_value is None or pd.isna(path_value):
        return None

    raw = str(path_value).strip()
    path = Path(raw)

    if path.is_absolute() and path.exists():
        return path

    candidate = PROJECT_ROOT / raw
    if candidate.exists():
        return candidate

    if fallback_base is not None:
        candidate = Path(fallback_base) / raw
        if candidate.exists():
            return candidate

    normalized = raw.replace(chr(92), "/")
    for marker in ["outputs/", "data/", "approaches/"]:
        if marker in normalized:
            suffix = normalized.split(marker, 1)[1]
            candidate = PROJECT_ROOT / marker.rstrip("/") / suffix
            if candidate.exists():
                return candidate

    if "outputs/models/" in normalized and "/00_linear_svm/models/" in normalized:
        filename = normalized.rsplit("/", 1)[-1]
        candidate = MODEL_DIR / filename
        if candidate.exists():
            return candidate

    if "outputs/models/" in normalized and "/01_linear_svm/models/" in normalized:
        filename = normalized.rsplit("/", 1)[-1]
        candidate = MODEL_DIR / filename
        if candidate.exists():
            return candidate

    if raw.endswith(".joblib"):
        candidate = MODEL_DIR / Path(raw).name
        if candidate.exists():
            return candidate

    if raw.endswith(".parquet"):
        candidate = FEATURES_DIR / Path(raw).name
        if candidate.exists():
            return candidate

    return path


PROJECT_ROOT = find_project_root()

def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()

APPROACH_DIR = PROJECT_ROOT / "approaches" / "traditional_ml"

FEATURES_DIR = PROJECT_ROOT / "data" / "features" / DATASET_NAME
FEATURE_MANIFEST_PATH = FEATURES_DIR / "feature_manifest.csv"

MODEL_ROOT_DIR = PROJECT_ROOT / "outputs" / "models" / DATASET_NAME / "01_linear_svm"
MODEL_DIR = MODEL_ROOT_DIR
MODEL_REGISTRY_PATH = MODEL_ROOT_DIR / "model_registry.csv"
TRAINING_STATUS_PATH = MODEL_ROOT_DIR / "training_status.csv"

OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_paths = {
    "FEATURES_DIR": FEATURES_DIR,
    "FEATURE_MANIFEST_PATH": FEATURE_MANIFEST_PATH,
    "MODEL_ROOT_DIR": MODEL_ROOT_DIR,
    "MODEL_REGISTRY_PATH": MODEL_REGISTRY_PATH,
}

for name, path in required_paths.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} not found: {path}")

print(f"[OK] Project root: {PROJECT_ROOT}")
print(f"[OK] Features dir: {FEATURES_DIR}")
print(f"[OK] Model registry: {MODEL_REGISTRY_PATH}")
print(f"[OK] Model dir: {MODEL_DIR}")
print(f"[OK] Output dir: {OUTPUT_DIR}")


## 4. Kayıt ve Çıktı Yardımcıları

Tablo ve görsel kayıtları tek yerden yönetilir. Mevcut dosya varsa overwrite ayarına göre korunur veya yeniden yazılır.


In [ ]:
def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def log_output(message):
    print(message)


def log_section(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def log_dataframe(title, df, max_rows=REPORT_MAX_TABLE_ROWS):
    print()
    print(f"[DATAFRAME] {title}")
    print(f"Shape: {df.shape}")
    display(df.head(max_rows))


def log_figure(title, description, figure_path):
    figure_path = Path(figure_path)
    print(f"[FIGURE] {title}: {figure_path}")


def save_dataframe_csv(df, output_path, overwrite=False, index=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print(f"[SKIP] Existing CSV kept: {output_path}")
        try:
            existing = pd.read_csv(output_path)
            return existing, "loaded_existing"
        except pd.errors.EmptyDataError:
            print(f"[WARNING] Existing CSV is empty; current dataframe kept in memory: {output_path}")
            return df, "kept_current_dataframe"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing CSV: {output_path}")

    df.to_csv(output_path, index=index, encoding="utf-8-sig")
    print(f"[SAVE] CSV saved: {output_path}")
    return df, "created_or_overwritten"


def save_current_figure(figure_path, title, description="", overwrite=False, dpi=150):
    figure_path = Path(figure_path)
    figure_path.parent.mkdir(parents=True, exist_ok=True)

    if figure_path.exists() and not overwrite:
        plt.close()
        print(f"[SKIP] Existing figure kept: {figure_path}")
        log_figure(title, description, figure_path)
        return figure_path, "loaded_existing"

    if figure_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing figure: {figure_path}")

    plt.savefig(figure_path, dpi=dpi)
    plt.close()
    print(f"[SAVE] Figure saved: {figure_path}")
    log_figure(title, description, figure_path)
    return figure_path, "created_or_overwritten"


print("[OK] Kayıt ve çıktı yardımcıları hazır.")


## 5. Girdi Envanteri

Model registry, feature manifest ve varsa eğitim durum tablosu okunarak temel girdi kontrolü yapılır.


In [ ]:
log_section("01 INPUT INVENTORY")

model_registry_df = pd.read_csv(MODEL_REGISTRY_PATH)

feature_manifest_df = pd.read_csv(FEATURE_MANIFEST_PATH)

if TRAINING_STATUS_PATH.exists():
    training_status_df = pd.read_csv(TRAINING_STATUS_PATH)
else:
    training_status_df = pd.DataFrame()
    log_output(f"[WARNING] Training status file not found: {TRAINING_STATUS_PATH}")

input_inventory_df = pd.DataFrame([
    {
        "input_name": "model_registry",
        "path": relative_path(MODEL_REGISTRY_PATH),
        "exists": MODEL_REGISTRY_PATH.exists(),
        "rows": len(model_registry_df),
        "columns": len(model_registry_df.columns),
    },
    {
        "input_name": "feature_manifest",
        "path": relative_path(FEATURE_MANIFEST_PATH),
        "exists": FEATURE_MANIFEST_PATH.exists(),
        "rows": len(feature_manifest_df),
        "columns": len(feature_manifest_df.columns),
    },
    {
        "input_name": "training_status",
        "path": relative_path(TRAINING_STATUS_PATH),
        "exists": TRAINING_STATUS_PATH.exists(),
        "rows": len(training_status_df),
        "columns": len(training_status_df.columns) if not training_status_df.empty else 0,
    },
])

save_dataframe_csv(input_inventory_df, TABLES_DIR / "input_inventory.csv", overwrite=OVERWRITE_TABLES, note="Stage 05 input inventory")
log_dataframe("Input Inventory", input_inventory_df)
log_dataframe("Model Registry Preview", model_registry_df.head(DISPLAY_PREVIEW_ROWS))
log_dataframe("Feature Manifest Preview", feature_manifest_df.head(DISPLAY_PREVIEW_ROWS))
if not training_status_df.empty:
    log_dataframe("Training Status", training_status_df)


## 6. Registry ve Feature Yolu Kontrolü

Model registry içindeki model ve feature yolları mevcut klasör yapısına göre çözülür; validasyon planı oluşturulur.


In [ ]:
log_section("02 REGISTRY AND FEATURE PATH VALIDATION")


def parse_patch_size(patch_set):
    match = re.search(r"centered(\d+)_", str(patch_set))
    if match:
        return int(match.group(1))
    return np.nan


def parse_ratio_variant(patch_set):
    patch_set = str(patch_set)
    if patch_set.endswith("_balanced"):
        return "balanced"
    if patch_set.endswith("_neg4x"):
        return "neg4x"
    return "unknown"


def classify_feature_family(feature_set):
    feature_set = str(feature_set)
    if feature_set == "hsv_lbp_only":
        return "hsv_lbp_only"
    if "pca" in feature_set:
        return "hog_pca_combined" if ("hsv" in feature_set or "lbp" in feature_set) else "hog_pca_only"
    if feature_set.endswith("_only") and feature_set.startswith("hog"):
        return "hog_only"
    if "hsv" in feature_set or "lbp" in feature_set:
        return "hog_hsv_lbp_combined"
    return "other"


def get_feature_path_from_registry(row):
    if "feature_file_path" in row.index and pd.notna(row["feature_file_path"]):
        candidate = resolve_project_path(row["feature_file_path"], fallback_base=FEATURES_DIR)
        if candidate is not None and Path(candidate).exists():
            return Path(candidate)

    patch_set = row["patch_set"]
    feature_set = row["feature_set"]
    candidate = FEATURES_DIR / f"{patch_set}__{feature_set}.parquet"
    return candidate


def get_model_path_from_registry(row):
    if "model_file_path" in row.index and pd.notna(row["model_file_path"]):
        candidate = resolve_project_path(row["model_file_path"], fallback_base=MODEL_DIR)
        if candidate is not None and Path(candidate).exists():
            return Path(candidate)

    model_id = row["model_id"]
    patch_set = row["patch_set"]
    feature_set = row["feature_set"]
    candidate = MODEL_DIR / f"{model_id}__{patch_set}__{feature_set}.joblib"
    return candidate


required_registry_columns = ["model_id", "dataset_name", "patch_set", "feature_set"]
missing_registry_columns = [col for col in required_registry_columns if col not in model_registry_df.columns]
if missing_registry_columns:
    raise ValueError(f"Model registry is missing required columns: {missing_registry_columns}")

validation_rows = []
validation_plan_rows = []

for _, row in model_registry_df.iterrows():
    patch_set = str(row["patch_set"])
    feature_set = str(row["feature_set"])
    model_id = str(row["model_id"])

    model_path = get_model_path_from_registry(row)
    feature_path = get_feature_path_from_registry(row)

    model_exists = Path(model_path).exists()
    feature_exists = Path(feature_path).exists()

    validation_rows.append({
        "dataset_name": DATASET_NAME,
        "model_id": model_id,
        "patch_set": patch_set,
        "feature_set": feature_set,
        "model_file_path": relative_path(model_path),
        "model_exists": model_exists,
        "feature_file_path": relative_path(feature_path),
        "feature_exists": feature_exists,
    })

    validation_plan_rows.append({
        "dataset_name": DATASET_NAME,
        "model_id": model_id,
        "patch_set": patch_set,
        "patch_size": parse_patch_size(patch_set),
        "ratio_variant": parse_ratio_variant(patch_set),
        "feature_set": feature_set,
        "feature_family": classify_feature_family(feature_set),
        "model_file_path": relative_path(model_path),
        "feature_file_path": relative_path(feature_path),
        "evaluation_split": EVALUATION_SPLIT,
    })

registry_path_validation_df = pd.DataFrame(validation_rows)
validation_plan_df = pd.DataFrame(validation_plan_rows)

missing_model_count = int((~registry_path_validation_df["model_exists"]).sum())
missing_feature_count = int((~registry_path_validation_df["feature_exists"]).sum())

if missing_model_count > 0 or missing_feature_count > 0:
    log_dataframe("Registry Path Validation Problems", registry_path_validation_df.query("model_exists == False or feature_exists == False"))
    raise FileNotFoundError(
        f"Missing model files: {missing_model_count}; missing feature files: {missing_feature_count}. "
        "Stage 05 cannot continue until official Stage 04 and Stage 03 files are available."
    )

if len(validation_plan_df) != 78:
    log_output(f"[WARNING] Expected 78 models for this dataset, found {len(validation_plan_df)}.")

save_dataframe_csv(registry_path_validation_df, TABLES_DIR / "registry_path_validation.csv", overwrite=OVERWRITE_TABLES, note="registry path validation")
save_dataframe_csv(validation_plan_df, TABLES_DIR / "validation_plan.csv", overwrite=OVERWRITE_TABLES, note="valid split validation plan")

log_dataframe("Registry Path Validation", registry_path_validation_df)
log_dataframe("Validation Plan", validation_plan_df)


## 7. Değerlendirme Yardımcıları

Model yükleme, valid split seçimi, tahmin üretimi ve metrik hesaplama fonksiyonları tanımlanır.


In [ ]:
log_section("03 EVALUATION HELPER FUNCTIONS")

METADATA_COLUMNS = {
    "patch_id",
    "dataset_name",
    "split",
    "patch_set",
    "patch_size",
    "ratio_variant",
    "patch_label",
    "patch_type",
    "source_image_path",
    "x1",
    "y1",
    "x2",
    "y2",
    "crop_width",
    "crop_height",
    "image_width",
    "image_height",
    "negative_type",
    "requested_negative_zone",
    "actual_negative_zone",
    "actual_normalized_center_distance",
    "was_shifted_to_fit_image",
    "source_bbox_class_id",
    "source_bbox_class_name",
    "source_bbox_width_px",
    "source_bbox_height_px",
    "source_bbox_area_px",
    "flags",
}


def extract_estimator(loaded_object):
    """Return a fitted estimator or pipeline from common joblib payload formats."""
    if hasattr(loaded_object, "predict"):
        return loaded_object

    if isinstance(loaded_object, dict):
        candidate_keys = [
            "estimator",
            "pipeline",
            "model",
            "best_estimator",
            "best_estimator_",
            "trained_model",
        ]
        for key in candidate_keys:
            if key in loaded_object and hasattr(loaded_object[key], "predict"):
                return loaded_object[key]

    raise TypeError(
        "Loaded joblib object does not look like a fitted estimator or supported payload dictionary."
    )


def get_feature_columns(feature_df, estimator=None):
    """Infer feature columns, preferring estimator feature_names_in_ if available."""
    if estimator is not None and hasattr(estimator, "feature_names_in_"):
        feature_names = list(estimator.feature_names_in_)
        missing = [col for col in feature_names if col not in feature_df.columns]
        if missing:
            raise ValueError(f"Feature file is missing estimator feature_names_in_ columns: {missing[:10]}")
        return feature_names

    return [
        col for col in feature_df.columns
        if col not in METADATA_COLUMNS
    ]


def load_valid_feature_data(feature_path, estimator=None):
    feature_df = pd.read_parquet(feature_path)
    if "split" not in feature_df.columns:
        raise ValueError(f"Feature file has no split column: {feature_path}")
    if "patch_label" not in feature_df.columns:
        raise ValueError(f"Feature file has no patch_label column: {feature_path}")

    valid_df = feature_df[feature_df["split"].astype(str) == EVALUATION_SPLIT].copy()
    if valid_df.empty:
        raise ValueError(f"No valid rows found in feature file: {feature_path}")

    feature_columns = get_feature_columns(valid_df, estimator=estimator)
    if not feature_columns:
        raise ValueError(f"No feature columns detected in feature file: {feature_path}")

    X_valid = valid_df[feature_columns]
    y_valid = valid_df["patch_label"].astype(int).to_numpy()

    return valid_df, X_valid, y_valid, feature_columns


def compute_binary_metrics(y_true, y_pred):
    labels = [0, 1]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "valid_accuracy": accuracy_score(y_true, y_pred),
        "valid_precision": precision_score(y_true, y_pred, zero_division=0),
        "valid_recall": recall_score(y_true, y_pred, zero_division=0),
        "valid_f1": f1_score(y_true, y_pred, zero_division=0),
        "valid_specificity": specificity,
        "valid_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "valid_tp": int(tp),
        "valid_fp": int(fp),
        "valid_tn": int(tn),
        "valid_fn": int(fn),
    }


def compute_score_summary(y_true, decision_scores):
    y_true = np.asarray(y_true)
    decision_scores = np.asarray(decision_scores, dtype=np.float64)

    positive_scores = decision_scores[y_true == 1]
    negative_scores = decision_scores[y_true == 0]

    def safe_mean(values):
        return float(np.mean(values)) if len(values) else np.nan

    def safe_std(values):
        return float(np.std(values)) if len(values) else np.nan

    def safe_median(values):
        return float(np.median(values)) if len(values) else np.nan

    def safe_quantile(values, q):
        return float(np.quantile(values, q)) if len(values) else np.nan

    positive_mean = safe_mean(positive_scores)
    negative_mean = safe_mean(negative_scores)

    return {
        "positive_score_mean": positive_mean,
        "negative_score_mean": negative_mean,
        "score_separation_mean": positive_mean - negative_mean if not (pd.isna(positive_mean) or pd.isna(negative_mean)) else np.nan,
        "positive_score_std": safe_std(positive_scores),
        "negative_score_std": safe_std(negative_scores),
        "positive_score_median": safe_median(positive_scores),
        "negative_score_median": safe_median(negative_scores),
        "score_separation_median": safe_median(positive_scores) - safe_median(negative_scores) if len(positive_scores) and len(negative_scores) else np.nan,
        "positive_score_q25": safe_quantile(positive_scores, 0.25),
        "positive_score_q75": safe_quantile(positive_scores, 0.75),
        "negative_score_q25": safe_quantile(negative_scores, 0.25),
        "negative_score_q75": safe_quantile(negative_scores, 0.75),
        "positive_score_min": float(np.min(positive_scores)) if len(positive_scores) else np.nan,
        "positive_score_max": float(np.max(positive_scores)) if len(positive_scores) else np.nan,
        "negative_score_min": float(np.min(negative_scores)) if len(negative_scores) else np.nan,
        "negative_score_max": float(np.max(negative_scores)) if len(negative_scores) else np.nan,
    }


def evaluate_one_model(plan_row):
    model_path = Path(plan_row["model_file_path"])
    feature_path = Path(plan_row["feature_file_path"])

    loaded_model = joblib.load(model_path)
    estimator = extract_estimator(loaded_model)

    valid_df, X_valid, y_valid, feature_columns = load_valid_feature_data(feature_path, estimator=estimator)

    y_pred = estimator.predict(X_valid).astype(int)

    if hasattr(estimator, "decision_function"):
        decision_scores = estimator.decision_function(X_valid)
    elif hasattr(estimator, "predict_proba"):
        decision_scores = estimator.predict_proba(X_valid)[:, 1]
    else:
        decision_scores = y_pred.astype(float)

    metric_values = compute_binary_metrics(y_valid, y_pred)
    score_values = compute_score_summary(y_valid, decision_scores)

    base = {
        "dataset_name": DATASET_NAME,
        "model_id": plan_row["model_id"],
        "patch_set": plan_row["patch_set"],
        "patch_size": plan_row["patch_size"],
        "ratio_variant": plan_row["ratio_variant"],
        "feature_set": plan_row["feature_set"],
        "feature_family": plan_row["feature_family"],
        "evaluation_split": EVALUATION_SPLIT,
        "valid_row_count": int(len(valid_df)),
        "valid_positive_count": int((y_valid == 1).sum()),
        "valid_negative_count": int((y_valid == 0).sum()),
        "feature_column_count": int(len(feature_columns)),
        "model_file_path": relative_path(model_path),
        "feature_file_path": relative_path(feature_path),
    }

    metrics_row = {**base, **metric_values}
    score_row = {**base, **score_values}

    confusion_rows = [
        {**base, "actual_label": 0, "predicted_label": 0, "count": int(metric_values["valid_tn"]), "cell": "tn"},
        {**base, "actual_label": 0, "predicted_label": 1, "count": int(metric_values["valid_fp"]), "cell": "fp"},
        {**base, "actual_label": 1, "predicted_label": 0, "count": int(metric_values["valid_fn"]), "cell": "fn"},
        {**base, "actual_label": 1, "predicted_label": 1, "count": int(metric_values["valid_tp"]), "cell": "tp"},
    ]

    score_detail_df = pd.DataFrame({
        "dataset_name": DATASET_NAME,
        "model_id": plan_row["model_id"],
        "patch_set": plan_row["patch_set"],
        "feature_set": plan_row["feature_set"],
        "ratio_variant": plan_row["ratio_variant"],
        "patch_label": y_valid,
        "predicted_label": y_pred,
        "decision_score": decision_scores,
    })

    return metrics_row, score_row, confusion_rows, score_detail_df


print("[OK] Evaluation helper functions are ready.")


## 8. Valid Split Model Değerlendirmesi

Valid split üzerinde her Linear SVM modeli için metrikler, skor özetleri ve confusion matrix tabloları oluşturulur.


In [ ]:
log_section("04 VALID SPLIT MODEL EVALUATION")

metric_rows = []
score_summary_rows = []
confusion_matrix_rows = []
score_detail_frames = []
evaluation_status_rows = []

for index, plan_row in validation_plan_df.iterrows():
    model_id = plan_row["model_id"]
    patch_set = plan_row["patch_set"]
    feature_set = plan_row["feature_set"]

    print(f"[RUN] {index + 1:03d}/{len(validation_plan_df):03d} — {model_id} — {patch_set} — {feature_set}")

    try:
        metrics_row, score_row, confusion_rows, score_detail_df = evaluate_one_model(plan_row)

        metric_rows.append(metrics_row)
        score_summary_rows.append(score_row)
        confusion_matrix_rows.extend(confusion_rows)
        score_detail_frames.append(score_detail_df)

        evaluation_status_rows.append({
            "dataset_name": DATASET_NAME,
            "model_id": model_id,
            "patch_set": patch_set,
            "feature_set": feature_set,
            "status": "OK",
            "error": "",
        })

    except Exception as exc:
        evaluation_status_rows.append({
            "dataset_name": DATASET_NAME,
            "model_id": model_id,
            "patch_set": patch_set,
            "feature_set": feature_set,
            "status": "FAILED",
            "error": repr(exc),
        })
        print(f"[ERROR] Evaluation failed for {model_id}: {repr(exc)}")
        raise

valid_model_metrics_df = pd.DataFrame(metric_rows)
valid_score_summary_df = pd.DataFrame(score_summary_rows)
valid_confusion_matrices_df = pd.DataFrame(confusion_matrix_rows)
evaluation_status_df = pd.DataFrame(evaluation_status_rows)
valid_score_details_df = pd.concat(score_detail_frames, ignore_index=True) if score_detail_frames else pd.DataFrame()

save_dataframe_csv(valid_model_metrics_df, TABLES_DIR / "valid_model_metrics.csv", overwrite=OVERWRITE_TABLES, note="valid split model metrics")
save_dataframe_csv(valid_score_summary_df, TABLES_DIR / "valid_score_summary.csv", overwrite=OVERWRITE_TABLES, note="valid split decision score summary")
save_dataframe_csv(valid_confusion_matrices_df, TABLES_DIR / "valid_confusion_matrices.csv", overwrite=OVERWRITE_TABLES, note="valid split confusion matrices")
save_dataframe_csv(evaluation_status_df, TABLES_DIR / "evaluation_status.csv", overwrite=OVERWRITE_TABLES, note="model evaluation status")
save_dataframe_csv(valid_score_details_df, TABLES_DIR / "valid_score_details_for_top_plotting.csv", overwrite=OVERWRITE_TABLES, note="decision scores used for score distribution plotting")

log_dataframe("Evaluation Status", evaluation_status_df)
log_dataframe("Valid Model Metrics", valid_model_metrics_df)
log_dataframe("Valid Score Summary", valid_score_summary_df)
log_dataframe("Valid Confusion Matrices", valid_confusion_matrices_df)


## 9. Sıralama ve İnceleme Listeleri

Modeller valid F1 metriğine göre sıralanır; genel ve oran bazlı kısa inceleme listeleri hazırlanır.


In [ ]:
log_section("05 RANKING AND REVIEW LISTS")

rank_sort_columns = [
    "valid_f1",
    "valid_recall",
    "valid_precision",
    "valid_specificity",
    "valid_balanced_accuracy",
    "valid_fp",
    "valid_fn",
]
rank_sort_ascending = [False, False, False, False, False, True, True]

ranked_linear_svm_models_df = (
    valid_model_metrics_df
    .sort_values(rank_sort_columns, ascending=rank_sort_ascending)
    .reset_index(drop=True)
)
ranked_linear_svm_models_df.insert(0, "valid_rank", np.arange(1, len(ranked_linear_svm_models_df) + 1))

overall_top20_by_valid_f1_df = ranked_linear_svm_models_df.head(OVERALL_TOP_N).copy()
overall_top20_by_valid_f1_df.insert(1, "review_list_type", "overall_top20_by_valid_f1")
overall_top20_by_valid_f1_df.insert(2, "review_rank", np.arange(1, len(overall_top20_by_valid_f1_df) + 1))

balanced_top10_by_valid_f1_df = (
    ranked_linear_svm_models_df
    .query("ratio_variant == 'balanced'")
    .head(RATIO_TOP_N)
    .copy()
)
balanced_top10_by_valid_f1_df.insert(1, "review_list_type", "balanced_top10_by_valid_f1")
balanced_top10_by_valid_f1_df.insert(2, "review_rank", np.arange(1, len(balanced_top10_by_valid_f1_df) + 1))

neg4x_top10_by_valid_f1_df = (
    ranked_linear_svm_models_df
    .query("ratio_variant == 'neg4x'")
    .head(RATIO_TOP_N)
    .copy()
)
neg4x_top10_by_valid_f1_df.insert(1, "review_list_type", "neg4x_top10_by_valid_f1")
neg4x_top10_by_valid_f1_df.insert(2, "review_rank", np.arange(1, len(neg4x_top10_by_valid_f1_df) + 1))

linear_svm_review_shortlist_df = pd.concat(
    [
        overall_top20_by_valid_f1_df,
        balanced_top10_by_valid_f1_df,
        neg4x_top10_by_valid_f1_df,
    ],
    ignore_index=True,
)

linear_svm_review_shortlist_df["selected_by_user"] = False
linear_svm_review_shortlist_df["selected_for_stage06"] = False
linear_svm_review_shortlist_df["selection_reason"] = ""

# This template is intentionally empty. It will be filled only after user review.
selected_linear_candidates_template_df = linear_svm_review_shortlist_df.head(0).copy()

save_dataframe_csv(ranked_linear_svm_models_df, TABLES_DIR / "ranked_linear_svm_models.csv", overwrite=OVERWRITE_TABLES, note="all official models ranked by valid_f1")
save_dataframe_csv(overall_top20_by_valid_f1_df, TABLES_DIR / "overall_top20_by_valid_f1.csv", overwrite=OVERWRITE_TABLES, note="overall top 20 review list")
save_dataframe_csv(balanced_top10_by_valid_f1_df, TABLES_DIR / "balanced_top10_by_valid_f1.csv", overwrite=OVERWRITE_TABLES, note="balanced top 10 review list")
save_dataframe_csv(neg4x_top10_by_valid_f1_df, TABLES_DIR / "neg4x_top10_by_valid_f1.csv", overwrite=OVERWRITE_TABLES, note="neg4x top 10 review list")
save_dataframe_csv(linear_svm_review_shortlist_df, TABLES_DIR / "linear_svm_review_shortlist.csv", overwrite=OVERWRITE_TABLES, note="combined review shortlist")
save_dataframe_csv(selected_linear_candidates_template_df, TABLES_DIR / "selected_linear_candidates_TEMPLATE.csv", overwrite=OVERWRITE_TABLES, note="empty template to be filled after user review")

log_dataframe("Ranked LinearSVM Models", ranked_linear_svm_models_df)
log_dataframe("Overall Top 20 by Valid F1", overall_top20_by_valid_f1_df)
log_dataframe("Balanced Top 10 by Valid F1", balanced_top10_by_valid_f1_df)
log_dataframe("Neg4x Top 10 by Valid F1", neg4x_top10_by_valid_f1_df)
log_dataframe("LinearSVM Review Shortlist", linear_svm_review_shortlist_df)


## 10. Özet Tablolar

Ratio variant, patch set ve feature ailesi bazında özet performans tabloları kaydedilir.


In [ ]:
log_section("06 SUMMARY TABLES")

ratio_summary_df = (
    ranked_linear_svm_models_df
    .groupby("ratio_variant", as_index=False)
    .agg(
        model_count=("model_id", "count"),
        mean_valid_f1=("valid_f1", "mean"),
        median_valid_f1=("valid_f1", "median"),
        max_valid_f1=("valid_f1", "max"),
        mean_valid_precision=("valid_precision", "mean"),
        mean_valid_recall=("valid_recall", "mean"),
        mean_valid_specificity=("valid_specificity", "mean"),
        mean_valid_fp=("valid_fp", "mean"),
        mean_valid_fn=("valid_fn", "mean"),
    )
    .sort_values("max_valid_f1", ascending=False)
)

patch_set_summary_df = (
    ranked_linear_svm_models_df
    .groupby(["patch_set", "patch_size", "ratio_variant"], as_index=False)
    .agg(
        model_count=("model_id", "count"),
        mean_valid_f1=("valid_f1", "mean"),
        median_valid_f1=("valid_f1", "median"),
        max_valid_f1=("valid_f1", "max"),
        best_model_id=("model_id", "first"),
    )
    .sort_values("max_valid_f1", ascending=False)
)

feature_family_summary_df = (
    ranked_linear_svm_models_df
    .groupby("feature_family", as_index=False)
    .agg(
        model_count=("model_id", "count"),
        mean_valid_f1=("valid_f1", "mean"),
        median_valid_f1=("valid_f1", "median"),
        max_valid_f1=("valid_f1", "max"),
        best_model_id=("model_id", "first"),
    )
    .sort_values("max_valid_f1", ascending=False)
)

top_model_row = ranked_linear_svm_models_df.iloc[0].to_dict()

stage05_review_policy_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "evaluation_split": EVALUATION_SPLIT,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "overall_review_list": f"top {OVERALL_TOP_N} by valid_f1",
    "balanced_review_list": f"top {RATIO_TOP_N} balanced by valid_f1",
    "neg4x_review_list": f"top {RATIO_TOP_N} neg4x by valid_f1",
    "automatic_final_selection": False,
    "final_selection_rule": "User will review the reported lists and provide selected model IDs and rationale before Stage 06.",
}])

save_dataframe_csv(ratio_summary_df, TABLES_DIR / "ratio_variant_summary.csv", overwrite=OVERWRITE_TABLES, note="ratio variant performance summary")
save_dataframe_csv(patch_set_summary_df, TABLES_DIR / "patch_set_summary.csv", overwrite=OVERWRITE_TABLES, note="patch set performance summary")
save_dataframe_csv(feature_family_summary_df, TABLES_DIR / "feature_family_summary.csv", overwrite=OVERWRITE_TABLES, note="feature family performance summary")
save_dataframe_csv(stage05_review_policy_df, TABLES_DIR / "stage05_review_policy.csv", overwrite=OVERWRITE_TABLES, note="Stage 05 review policy")

log_dataframe("Ratio Variant Summary", ratio_summary_df)
log_dataframe("Patch Set Summary", patch_set_summary_df)
log_dataframe("Feature Family Summary", feature_family_summary_df)
log_dataframe("Stage 05 Review Policy", stage05_review_policy_df)


## 11. Validasyon Görselleri

Model sıralaması, precision/recall dağılımı ve hata sayıları için görseller oluşturulur.


In [ ]:
log_section("07 VALIDATION FIGURES")

# Figure 1 — valid F1 by ranked model
plot_df = ranked_linear_svm_models_df.sort_values("valid_rank").copy()
plt.figure(figsize=(14, 5))
plt.bar(plot_df["model_id"], plot_df["valid_f1"])
plt.xticks(rotation=90, ha="center")
plt.ylabel("Valid F1")
plt.xlabel("Model ID")
plt.title(f"{DATASET_DISPLAY_NAME} — LinearSVM Valid F1 by Model")
plt.tight_layout()
valid_f1_fig_path = FIGURES_DIR / "valid_f1_by_model.png"
save_current_figure(valid_f1_fig_path, "Valid F1 by Model", "All LinearSVM models ranked by valid F1.", overwrite=OVERWRITE_FIGURES)

# Figure 2 — precision recall scatter
plt.figure(figsize=(7, 6))
for ratio_variant, group_df in ranked_linear_svm_models_df.groupby("ratio_variant"):
    plt.scatter(group_df["valid_recall"], group_df["valid_precision"], label=ratio_variant, alpha=0.8)
plt.xlabel("Valid Recall")
plt.ylabel("Valid Precision")
plt.title(f"{DATASET_DISPLAY_NAME} — Precision / Recall Scatter")
plt.legend()
plt.tight_layout()
precision_recall_fig_path = FIGURES_DIR / "precision_recall_scatter.png"
save_current_figure(precision_recall_fig_path, "Precision / Recall Scatter", "Valid precision and recall for all LinearSVM models.", overwrite=OVERWRITE_FIGURES)

# Figure 3 — top 20 F1 with FP/FN
top20_plot_df = overall_top20_by_valid_f1_df.copy()
x = np.arange(len(top20_plot_df))
width = 0.35

plt.figure(figsize=(14, 6))
plt.bar(x - width / 2, top20_plot_df["valid_fp"], width, label="FP")
plt.bar(x + width / 2, top20_plot_df["valid_fn"], width, label="FN")
plt.xticks(x, top20_plot_df["model_id"], rotation=75, ha="right")
plt.ylabel("Count")
plt.title(f"{DATASET_DISPLAY_NAME} — Top 20 FP / FN Counts")
plt.legend()
plt.tight_layout()
fp_fn_fig_path = FIGURES_DIR / "top20_fp_fn_counts.png"
save_current_figure(fp_fn_fig_path, "Top 20 FP / FN Counts", "False positive and false negative counts for the overall top 20 list.", overwrite=OVERWRITE_FIGURES)

# Figure 4 — valid F1 by ratio variant
plt.figure(figsize=(8, 5))
ratio_box_data = [
    ranked_linear_svm_models_df.loc[ranked_linear_svm_models_df["ratio_variant"] == ratio, "valid_f1"].to_numpy()
    for ratio in ["balanced", "neg4x"]
]
plt.boxplot(ratio_box_data, labels=["balanced", "neg4x"])
plt.ylabel("Valid F1")
plt.title(f"{DATASET_DISPLAY_NAME} — Valid F1 by Ratio Variant")
plt.tight_layout()
ratio_fig_path = FIGURES_DIR / "valid_f1_by_ratio_variant.png"
save_current_figure(ratio_fig_path, "Valid F1 by Ratio Variant", "Balanced and neg4x model distributions on valid F1.", overwrite=OVERWRITE_FIGURES)

# Figure 5 — decision score distributions for top 5 models
top_score_models = overall_top20_by_valid_f1_df["model_id"].head(5).tolist()
score_plot_df = valid_score_details_df[valid_score_details_df["model_id"].isin(top_score_models)].copy()

if not score_plot_df.empty:
    plt.figure(figsize=(12, 6))
    positions = []
    labels = []
    data = []
    pos = 1
    for model_id in top_score_models:
        model_df = score_plot_df[score_plot_df["model_id"] == model_id]
        negative_scores = model_df.loc[model_df["patch_label"] == 0, "decision_score"].to_numpy()
        positive_scores = model_df.loc[model_df["patch_label"] == 1, "decision_score"].to_numpy()
        data.extend([negative_scores, positive_scores])
        positions.extend([pos, pos + 0.35])
        labels.extend([f"{model_id}\nneg", f"{model_id}\npos"])
        pos += 1.2

    plt.boxplot(data, positions=positions, widths=0.28)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(positions, labels, rotation=75, ha="right")
    plt.ylabel("Decision score")
    plt.title(f"{DATASET_DISPLAY_NAME} — Top Model Decision Score Distributions")
    plt.tight_layout()
    score_dist_fig_path = FIGURES_DIR / "top_score_distributions.png"
    save_current_figure(score_dist_fig_path, "Top Model Decision Score Distributions", "Decision score distributions for positive and negative valid patches in the top five models.", overwrite=OVERWRITE_FIGURES)


## 12. Final Durum

Notebook sonunda validasyon özeti ve durum tablosu kaydedilir.


In [ ]:
log_section("08 FINAL DURUM")

notebook_finished_at = datetime.now()
elapsed_minutes = (notebook_finished_at - NOTEBOOK_STARTED_AT).total_seconds() / 60

failed_evaluation_count = int((evaluation_status_df["status"] != "OK").sum())
status = "READY_FOR_REVIEW" if failed_evaluation_count == 0 else "COMPLETED_WITH_WARNINGS"

top_model_row = ranked_linear_svm_models_df.iloc[0]

notebook_status_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "stage_name": STAGE_NAME,
    "started_at": NOTEBOOK_STARTED_AT.strftime("%Y-%m-%d %H:%M:%S"),
    "finished_at": notebook_finished_at.strftime("%Y-%m-%d %H:%M:%S"),
    "elapsed_minutes": round(elapsed_minutes, 2),
    "evaluation_split": EVALUATION_SPLIT,
    "evaluated_model_count": len(valid_model_metrics_df),
    "failed_evaluation_count": failed_evaluation_count,
    "overall_top_n": OVERALL_TOP_N,
    "ratio_top_n": RATIO_TOP_N,
    "top_model_id": top_model_row["model_id"],
    "top_model_patch_set": top_model_row["patch_set"],
    "top_model_feature_set": top_model_row["feature_set"],
    "top_model_valid_f1": round(float(top_model_row["valid_f1"]), 6),
    "status": status,
}])

notebook_status_df, _ = save_dataframe_csv(
    notebook_status_df,
    TABLES_DIR / "notebook_status.csv",
    overwrite=OVERWRITE_TABLES,
    note="notebook status",
)
log_dataframe("Notebook Status", notebook_status_df)

log_output(f"Status: {status}")
log_output(f"Evaluated model count: {len(valid_model_metrics_df)}")
log_output(f"Failed evaluation count: {failed_evaluation_count}")
log_output(f"Top model: {top_model_row['model_id']} | valid_f1={float(top_model_row['valid_f1']):.6f}")
